# PCL Detection — RoBERTa + class-weighted CE + threshold optimisation

**Minimal pipeline** (proven to reach ~0.57 F1):
- **RoBERTa-base** (same as task baseline)
- **Raw text** (no keyword prepending)
- **Class-weighted cross-entropy** (pos_weight = n_neg/n_pos)
- **Threshold optimisation** on dev set (search in [0.3, 0.7])
- **Two-stage**: train → pick best epoch & threshold → retrain on train+dev → predict test

Baseline: 0.48 dev / 0.49 test. Target: beat that and aim for ~0.55+ with this setup.

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score, classification_report
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# Config — RoBERTa only, no toggles
CFG = {
    'model_name':  'roberta-base',
    'main_data':   'dontpatronizeme_pcl.tsv',
    'train_ids':   'train_semeval_parids-labels.csv',
    'dev_ids':     'dev_semeval_parids-labels.csv',
    'test_data':   'task4_test.tsv',
    'max_length':  256,
    'batch_size':  16,
    'grad_accum':  2,
    'lr':          2e-5,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'num_epochs':   8,
    'patience':     3,
    'seed':         42,
    'best_model':   'BestModel/best_model_roberta.pt',
    'final_model':  'BestModel/final_model_roberta.pt',
    'dev_preds':    'BestModel/dev.txt',
    'test_preds':   'BestModel/test.txt',
}
os.makedirs('BestModel', exist_ok=True)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
set_seed(CFG['seed'])

In [ ]:
# Data loading (same as main notebook, raw text only)
def load_main_data(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip('\n').split('\t', 5)
            if len(parts) == 6 and parts[0].strip().isdigit():
                rows.append({'par_id': int(parts[0]), 'keyword': parts[2], 'country': parts[3], 'text': parts[4].strip(), 'label_raw': int(parts[5])})
    df = pd.DataFrame(rows)
    df['label_binary'] = (df['label_raw'] >= 2).astype(int)
    return df

def load_split_ids(path):
    df = pd.read_csv(path)
    df['par_id'] = df['par_id'].astype(int)
    return df[['par_id']]

def load_test_data(path):
    rows = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip('\n').split('\t')
            if len(parts) >= 5 and parts[0].strip().startswith('t_'):
                rows.append({'par_id': parts[0].strip(), 'text': parts[4].strip() if len(parts) > 4 else ''})
    df = pd.DataFrame(rows)
    return df[df['text'].str.len() > 0].reset_index(drop=True)

df_main = load_main_data(CFG['main_data'])
df_train_ids = load_split_ids(CFG['train_ids'])
df_dev_ids   = load_split_ids(CFG['dev_ids'])
df_test      = load_test_data(CFG['test_data'])

df_train = df_main[df_main['par_id'].isin(df_train_ids['par_id'])].copy()
df_dev   = df_main[df_main['par_id'].isin(df_dev_ids['par_id'])].copy()

n_pos = (df_train['label_binary'] == 1).sum()
n_neg = (df_train['label_binary'] == 0).sum()
print(f"Train: {len(df_train)}, Dev: {len(df_dev)}, Test: {len(df_test)}")
print(f"Train PCL: {n_pos}, No-PCL: {n_neg}  ratio {n_neg/n_pos:.2f}:1")

In [ ]:
# Dataset and loaders — binary only
class PCLDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = [str(t) for t in texts]
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], max_length=self.max_length, padding='max_length', truncation=True, return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(float(self.labels[idx]), dtype=torch.float)
        return item

tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])

train_ds = PCLDataset(df_train['text'].tolist(), df_train['label_binary'].tolist(), tokenizer, CFG['max_length'])
dev_ds   = PCLDataset(df_dev['text'].tolist(),   df_dev['label_binary'].tolist(),   tokenizer, CFG['max_length'])
test_ds  = PCLDataset(df_test['text'].tolist(),  [0]*len(df_test), tokenizer, CFG['max_length'])

# Oversample PCL so ~25% of each batch is positive
TARGET_POS = 0.25
w_pos = TARGET_POS / n_pos
w_neg = (1 - TARGET_POS) / n_neg
weights = np.where(df_train['label_binary'].values == 1, w_pos, w_neg)
sampler = WeightedRandomSampler(torch.FloatTensor(weights), len(weights), replacement=True)

train_dl = DataLoader(train_ds, batch_size=CFG['batch_size'], sampler=sampler, num_workers=0, pin_memory=True)
dev_dl   = DataLoader(dev_ds,   batch_size=CFG['batch_size']*2, shuffle=False, num_workers=0)
test_dl  = DataLoader(test_ds,  batch_size=CFG['batch_size']*2, shuffle=False, num_workers=0)

# Class-weighted CE: weight for positive class = n_neg/n_pos (full ratio)
pos_weight = torch.tensor([n_neg / n_pos], dtype=torch.float).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
print(f"pos_weight = {pos_weight.item():.2f}")

In [ ]:
# Helpers: RoBERTa outputs (batch, 2); we use logits[:, 1] as PCL logit
def run_epoch(model, loader, optimizer, scheduler, criterion, grad_accum):
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()
    for step, batch in enumerate(tqdm(loader, desc='Train', leave=False)):
        logits = model(batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device)).logits
        loss = criterion(logits[:, 1], batch['labels'].to(device)) / grad_accum
        loss.backward()
        if (step + 1) % grad_accum == 0 or (step + 1) == len(loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
        total_loss += loss.item() * grad_accum
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, threshold=0.5):
    model.eval()
    all_logits, all_labels = [], []
    for batch in tqdm(loader, desc='Eval', leave=False):
        logits = model(batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device)).logits
        all_logits.append(logits[:, 1].cpu().numpy())
        all_labels.append(batch['labels'].numpy())
    all_logits = np.concatenate(all_logits)
    all_labels = np.concatenate(all_labels)
    probs = 1.0 / (1.0 + np.exp(-all_logits))
    preds = (probs >= threshold).astype(int)
    return f1_score(all_labels.astype(int), preds, zero_division=0), probs, all_labels

def tune_threshold(probs, labels, t_min=0.3, t_max=0.7):
    best_t, best_f = 0.5, 0.0
    for t in np.linspace(t_min, t_max, 41):
        f = f1_score(labels.astype(int), (probs >= t).astype(int), zero_division=0)
        if f > best_f: best_f, best_t = f, t
    print(f"Best threshold: {best_t:.3f}  →  dev F1 = {best_f:.4f}")
    return float(best_t)

@torch.no_grad()
def predict(model, loader, threshold):
    model.eval()
    probs = []
    for batch in tqdm(loader, desc='Predict', leave=False):
        logits = model(batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device)).logits
        probs.append(torch.sigmoid(logits[:, 1]).cpu().numpy())
    probs = np.concatenate(probs)
    return (probs >= threshold).astype(int), probs

In [ ]:
# Stage 1: train on train set, early stop, tune threshold on dev
model = AutoModelForSequenceClassification.from_pretrained(CFG['model_name'], num_labels=2).to(device)
total_steps = (len(train_dl) // CFG['grad_accum']) * CFG['num_epochs']
optimizer = AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = get_linear_schedule_with_warmup(optimizer, int(total_steps*CFG['warmup_ratio']), total_steps)

best_f1, best_epoch, best_probs, best_labels = 0.0, 0, None, None
patience_cnt = 0

for epoch in range(CFG['num_epochs']):
    print(f"\n── Epoch {epoch+1}/{CFG['num_epochs']} ──")
    train_loss = run_epoch(model, train_dl, optimizer, scheduler, criterion, CFG['grad_accum'])
    f1, probs, labels = evaluate(model, dev_dl, threshold=0.5)
    print(f"   Train loss: {train_loss:.4f}  |  Dev F1 (t=0.5): {f1:.4f}", end='')
    if f1 > best_f1:
        best_f1, best_epoch, best_probs, best_labels = f1, epoch+1, probs.copy(), labels.copy()
        patience_cnt = 0
        torch.save(model.state_dict(), CFG['best_model'])
        print("  ✓ best")
    else:
        patience_cnt += 1
        print(f"  [patience {patience_cnt}/{CFG['patience']}]")
        if patience_cnt >= CFG['patience']:
            print("Early stop.")
            break

print(f"\nBest dev F1 = {best_f1:.4f} at epoch {best_epoch}")

In [ ]:
# Threshold optimisation on dev
model.load_state_dict(torch.load(CFG['best_model'], map_location=device))
_, best_probs, best_labels = evaluate(model, dev_dl, threshold=0.5)
best_thresh = tune_threshold(best_probs, best_labels)
preds_dev = (best_probs >= best_thresh).astype(int)
print(classification_report(best_labels.astype(int), preds_dev, target_names=['No-PCL', 'PCL'], digits=4))
with open(CFG['dev_preds'], 'w') as f:
    for p in preds_dev: f.write(f"{int(p)}\n")
print(f"Saved {CFG['dev_preds']}")

In [ ]:
# Stage 2: retrain on train+dev for best_epoch, then predict test
df_full = pd.concat([df_train, df_dev], ignore_index=True)
n_pos_f = (df_full['label_binary'] == 1).sum()
n_neg_f = (df_full['label_binary'] == 0).sum()
full_ds = PCLDataset(df_full['text'].tolist(), df_full['label_binary'].tolist(), tokenizer, CFG['max_length'])
w_pos_f = TARGET_POS / n_pos_f
w_neg_f = (1 - TARGET_POS) / n_neg_f
weights_f = np.where(df_full['label_binary'].values == 1, w_pos_f, w_neg_f)
full_dl = DataLoader(full_ds, batch_size=CFG['batch_size'], sampler=WeightedRandomSampler(torch.FloatTensor(weights_f), len(weights_f), replacement=True), num_workers=0)
pos_weight_f = torch.tensor([n_neg_f / n_pos_f], dtype=torch.float).to(device)
criterion_f = nn.BCEWithLogitsLoss(pos_weight=pos_weight_f)

model_final = AutoModelForSequenceClassification.from_pretrained(CFG['model_name'], num_labels=2).to(device)
steps_s2 = (len(full_dl) // CFG['grad_accum']) * best_epoch
opt_s2 = AdamW(model_final.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
sched_s2 = get_linear_schedule_with_warmup(opt_s2, int(steps_s2*CFG['warmup_ratio']), steps_s2)

for epoch in range(best_epoch):
    print(f"Stage 2 epoch {epoch+1}/{best_epoch}")
    run_epoch(model_final, full_dl, opt_s2, sched_s2, criterion_f, CFG['grad_accum'])

torch.save(model_final.state_dict(), CFG['final_model'])
preds_test, probs_test = predict(model_final, test_dl, threshold=best_thresh)
with open(CFG['test_preds'], 'w') as f:
    for p in preds_test: f.write(f"{int(p)}\n")
print(f"Saved {CFG['test_preds']}  (threshold={best_thresh:.3f})")